# Lab 1 - Pipeline de adquisicion y analisis de textos
Notebook de trabajo para capturar textos desde X y RSS, consolidarlos, preprocesarlos y analizar resultados.

## Librerias principales
pandas, requests, feedparser, pyarrow, nltk, scikit-learn, pymongo, matplotlib.

In [ ]:
%pip install -r ../requirements.txt

In [1]:
from pathlib import Path
import sys
from getpass import getpass
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "config.py").exists():
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from config import (
    X_QUERY,
    X_MAX_RESULTS,
    COOPERATIVA_RSS_URL,
    COOPERATIVA_MAX_ITEMS,
    PARQUET_OUTPUT_NAME,
    PICKLE_OUTPUT_NAME,
)
from procesamiento.x_api import buscar_posts_x
from procesamiento.rss import leer_rss
from fuentes.consolidacion import consolidar_dataframes
from fuentes.limpieza import limpiar_texto
from fuentes.analisis import obtener_frecuencias, aplicar_tfidf, aplicar_kmeans, aplicar_pca
from persistencia.binaria import guardar_binario

In [ ]:
bearer = getpass("Bearer Token de X (si aun no lo tienes, Enter temporal): ").strip()
if bearer:
    df_x = buscar_posts_x(X_QUERY, bearer, X_MAX_RESULTS)
else:
    print("[PENDIENTE] Ejecutando temporalmente sin X. Cuando tengas Bearer, vuelve a correr desde esta celda.")
    df_x = pd.DataFrame(columns=["id", "fuente", "texto", "fecha", "autor", "url", "consulta"])

df_coop = leer_rss(COOPERATIVA_RSS_URL, "Cooperativa", COOPERATIVA_MAX_ITEMS)

print(f"X: {len(df_x)} filas")
print(f"Cooperativa: {len(df_coop)} filas")

In [ ]:
df_corpus = consolidar_dataframes([df_x, df_coop])
if df_corpus.empty:
    raise ValueError("No se obtuvieron textos desde X y Cooperativa.")

df_corpus = df_corpus.reset_index(drop=True)
df_corpus.head()

In [ ]:
df_corpus["texto_limpio"] = df_corpus["texto"].apply(limpiar_texto)
frecuencias = obtener_frecuencias(df_corpus["texto_limpio"].tolist())
pd.DataFrame(frecuencias.most_common(20), columns=["token", "frecuencia"])

In [ ]:
X_tfidf, vectorizer = aplicar_tfidf(df_corpus)
df_corpus, modelo_kmeans = aplicar_kmeans(X_tfidf, df_corpus)
df_corpus = aplicar_pca(X_tfidf, df_corpus)

df_corpus[["fuente", "cluster", "pca_1", "pca_2"]].head()

In [ ]:
guardar_binario(df_corpus, PARQUET_OUTPUT_NAME, PICKLE_OUTPUT_NAME)
print(f"Parquet guardado en: {PARQUET_OUTPUT_NAME}")
print(f"Pickle guardado en: {PICKLE_OUTPUT_NAME}")

In [ ]:
from config import MONGO_DB_NAME, MONGO_COLLECTION_NAME
from persistencia.mongodb import guardar_en_mongodb

mongo_uri = getpass("Mongo URI (requerida para guardar en MongoDB): ").strip()
if not mongo_uri:
    print("[PENDIENTE] Aun no hay Mongo URI. Ejecuta esta celda cuando la tengas.")
else:
    resumen_mongo = guardar_en_mongodb(df_corpus, mongo_uri, MONGO_DB_NAME, MONGO_COLLECTION_NAME)
    print("Resumen MongoDB:", resumen_mongo)

## Nota
Con Bearer disponible, vuelve a ejecutar desde la celda 5 para incorporar X y luego guardar el corpus final en MongoDB.